# Export to Ollama (GGUF): SFT + DPO model

Turns the final **SFT + DPO** model (`outputs/dpo_qwen2.5-0.5b/merged/`) into a **GGUF** file (weights + tokenizer + config in one file) and registers it in Ollama, so it runs with `ollama run` on CPU or GPU without Python.

**Tags:** this model becomes **`:dpo`** and **`:latest`**; the earlier SFT model is kept as **`:sft`** (pushing without a tag would overwrite `:latest` and lose it).

**Steps:** 1. setup → 2. llama.cpp converter → 3. GGUF → 4. Modelfile → 5. register in Ollama → 6. test → 7. accuracy check → 8. publish to ollama.com.

**Safe to re-run, top to bottom or from any step after step 1:**
- every step skips work that's already done (`FORCE = True` redoes it),
- every `ollama` command streams its output into the notebook and has a time limit, so nothing spins silently for an hour,
- step 1 checks Ollama is actually running before anything else.

**Needs:** the Ollama app running, and `git`.

## 1. Setup: config, helpers, Ollama check
**Run this first after every kernel restart**: all imports and helpers live here, so later cells never hit a `NameError`.
- `QUANT = "q8_0"`: 8-bit, ~530 MB, almost no accuracy loss (`"bf16"` = full precision, ~950 MB).
- `BUILD_DIR`: `ollama create` reads the GGUF from a folder **outside OneDrive**. A freshly written 500 MB file inside OneDrive can be locked while it uploads, which makes `create` hang.

In [1]:
import json, os, shutil, subprocess, sys, threading, time, urllib.request
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS = REPO_ROOT / "notebooks"
from src.prompts import build_messages

RUN_NAME = "dpo_qwen2.5-0.5b"                   # final model: SFT + DPO (merged)
MERGED_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "merged"
RESULTS_DIR = NOTEBOOKS / "data" / "eval" / "results" / RUN_NAME

QUANT = "q8_0"
OLLAMA_NAME = "qwen2.5_0.5b-instruct-fc-Chat"
TAG = "dpo"                                      # this version; also copied to :latest
OLLAMA_USER = "Avahan"
PUBLIC_NAME = f"{OLLAMA_USER}/{OLLAMA_NAME}"     # must match the page on ollama.com
FORCE = False                                    # True = redo conversion / registration even if done

GGUF_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "gguf"                 # gitignored
GGUF_FILE = GGUF_DIR / f"{OLLAMA_NAME}-{QUANT}.gguf"
BUILD_DIR = Path.home() / ".ollama_build" / RUN_NAME                 # outside OneDrive
LLAMA_CPP = REPO_ROOT / "third_party" / "llama.cpp"                  # gitignored
OLLAMA_API = "http://localhost:11434"
for d in (GGUF_DIR, BUILD_DIR):
    d.mkdir(parents=True, exist_ok=True)


def run(cmd, timeout=600, cwd=None, quiet_every=5.0):
    """Run a command, stream its output into the notebook, kill it after `timeout` seconds,
    even if it hangs without printing anything. Progress bars are throttled to one line every few seconds."""
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace")
    last = [0.0]

    def pump():                                  # reads output in the background
        for line in proc.stdout:                 # text mode: '\r' progress updates arrive as lines
            line, now = line.strip(), time.time()
            if line and (now - last[0] > quiet_every or any(w in line.lower() for w in ("error", "success", "writing"))):
                print("  ", line[:160])
                last[0] = now

    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    try:
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        raise TimeoutError(f"{cmd[0]} {cmd[1]} still running after {timeout}s -> killed. "
                           "Restart Ollama (tray icon -> Quit, reopen) and re-run this cell.")
    reader.join(timeout=5)
    if proc.returncode != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {' '.join(map(str, cmd))}")


def ollama_models():
    """Local model names, e.g. {'qwen2.5_0.5b-instruct-fc-Chat:dpo', ...} (via the API: never hangs)."""
    with urllib.request.urlopen(f"{OLLAMA_API}/api/tags", timeout=10) as r:
        return {m["name"] for m in json.loads(r.read())["models"]}


def ollama_chat(messages, model=None, timeout=300):
    body = json.dumps({"model": model or f"{OLLAMA_NAME}:{TAG}", "messages": messages, "stream": False,
                       "options": {"temperature": 0, "repeat_penalty": 1.0,
                                   "num_predict": 512}}).encode()   # cap: a looping answer stops at 512 tokens
    req = urllib.request.Request(f"{OLLAMA_API}/api/chat", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read())["message"]["content"]


try:
    with urllib.request.urlopen(f"{OLLAMA_API}/api/version", timeout=5) as r:
        print("Ollama running, version", json.loads(r.read())["version"])
except Exception as e:
    raise SystemExit(f"Ollama isn't reachable ({e}). Start the Ollama app, then re-run this cell.")
assert MERGED_DIR.exists(), f"missing {MERGED_DIR}"
print("Model folder:", MERGED_DIR)
print("Local models:", sorted(m for m in ollama_models() if OLLAMA_NAME in m) or "none yet")

Ollama running, version 0.40.4
Model folder: c:\Users\Swapn\OneDrive\Documents\GitHub\SLM Finetuning\notebooks\outputs\dpo_qwen2.5-0.5b\merged
Local models: ['Avahan/qwen2.5_0.5b-instruct-fc-Chat:latest']


## 2. Get llama.cpp's converter
Only its Python script `convert_hf_to_gguf.py` is needed: no compiling.

⚠️ Don't `pip install -r` llama.cpp's requirements file: it can replace your CUDA PyTorch with a CPU-only build. If the script complains about one missing package, install just that one.

In [2]:
if not (LLAMA_CPP / "convert_hf_to_gguf.py").exists():
    run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA_CPP], timeout=900)
print("Converter ready:", (LLAMA_CPP / "convert_hf_to_gguf.py").exists())

Converter ready: True


## 3. Convert to GGUF
Reads the safetensors + config + tokenizer and writes one `.gguf`, quantized to `q8_0` in the same pass (~1 min).
**Skipped** if the GGUF already exists and is newer than the merged model.

In [3]:
weights = MERGED_DIR / "model.safetensors"
if GGUF_FILE.exists() and GGUF_FILE.stat().st_mtime > weights.stat().st_mtime and not FORCE:
    print("GGUF already up to date, skipping conversion")
else:
    run([sys.executable, LLAMA_CPP / "convert_hf_to_gguf.py", MERGED_DIR,
         "--outfile", GGUF_FILE, "--outtype", QUANT], timeout=1800, quiet_every=10)
print(f"✅ {GGUF_FILE.name}: {GGUF_FILE.stat().st_size / 1024**2:.0f} MB (bf16 weights: {weights.stat().st_size / 1024**2:.0f} MB)")

GGUF already up to date, skipping conversion
✅ qwen2.5_0.5b-instruct-fc-Chat-q8_0.gguf: 506 MB (bf16 weights: 942 MB)


## 4. Write the Modelfile
A **Modelfile** is Ollama's recipe:
- **`FROM`**: the GGUF file.
- **`TEMPLATE`**: Qwen's chat format (`<|im_start|>role ... <|im_end|>`), same as training. Like Qwen's own template, it adds Qwen's default system line when the caller sends no system message.
- **No `SYSTEM` prompt baked in** (same as the Hugging Face upload). The *caller* sends the tool-calling system prompt (`build_messages` / the gateway do). Baking it in would force every chat into tool mode.
- **`PARAMETER`**: `temperature 0` (greedy, like the eval), `repeat_penalty 1.0` (Ollama's default 1.1 punishes the `"`, `{`, `:` that JSON repeats constantly), context 4096, stop at `<|im_end|>`.

The GGUF and the Modelfile are then copied to `BUILD_DIR` (outside OneDrive) for step 5.

In [4]:
TEMPLATE = """{{- if not (and .Messages (eq (index .Messages 0).Role "system")) }}<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
{{ end }}{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
"""

modelfile = f'''FROM ./{GGUF_FILE.name}

TEMPLATE """{TEMPLATE}"""

PARAMETER temperature 0
PARAMETER repeat_penalty 1.0
PARAMETER num_ctx 4096
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
'''
(GGUF_DIR / "Modelfile").write_text(modelfile, encoding="utf-8")       # kept next to the GGUF for reference
(BUILD_DIR / "Modelfile").write_text(modelfile, encoding="utf-8")

staged = BUILD_DIR / GGUF_FILE.name
if not staged.exists() or staged.stat().st_size != GGUF_FILE.stat().st_size or FORCE:
    print("copying GGUF outside OneDrive ->", BUILD_DIR)
    shutil.copy2(GGUF_FILE, staged)
print("build folder ready:", sorted(p.name for p in BUILD_DIR.iterdir()))

copying GGUF outside OneDrive -> C:\Users\Swapn\.ollama_build\dpo_qwen2.5-0.5b
build folder ready: ['Modelfile', 'qwen2.5_0.5b-instruct-fc-Chat-q8_0.gguf']


## 5. Register it in Ollama
1. **Keep the SFT version:** if the local `:latest` is still the SFT model and there's no `:sft` yet, copy it to **`:sft`** first.
2. **`ollama create`** registers the DPO GGUF as **`:dpo`** (skipped if it exists, unless `FORCE`), then it's copied to **`:latest`**.

Copies are cheap: Ollama stores each weight file once and tags just point at it. Normally takes well under a minute; it's killed after 10 min with a hint if Ollama hangs.

In [5]:
models = ollama_models()
dpo, sft, latest = f"{OLLAMA_NAME}:{TAG}", f"{OLLAMA_NAME}:sft", f"{OLLAMA_NAME}:latest"

if sft not in models and latest in models and dpo not in models:     # latest is still the SFT model
    run(["ollama", "cp", latest, sft], timeout=120)
    print("kept the SFT model as", sft)

if dpo not in models or FORCE:
    run(["ollama", "create", dpo, "-f", "Modelfile"], cwd=BUILD_DIR, timeout=600)
else:
    print(dpo, "already registered, skipping create (FORCE = True to rebuild)")
run(["ollama", "cp", dpo, latest], timeout=120)

print("\nLocal tags:", sorted(m for m in ollama_models() if OLLAMA_NAME in m))
if sft not in ollama_models():
    print("⚠️  no :sft tag: the SFT model was already overwritten earlier. Recreate it from "
          "outputs/sft_qwen2.5-0.5b/gguf if you still want to publish it.")

$ ollama create qwen2.5_0.5b-instruct-fc-Chat:dpo -f Modelfile
   gathering model components ⠋ gathering model components ⠙ gathering mode
   writing manifest ⠋ gathering model components 
   writing manifest ⠙ gathering model components 
   writing manifest 
   success 
$ ollama cp qwen2.5_0.5b-instruct-fc-Chat:dpo qwen2.5_0.5b-instruct-fc-Chat:latest
   copied 'qwen2.5_0.5b-instruct-fc-Chat:dpo' to 'qwen2.5_0.5b-instruct-fc-Chat:latest'
   success

Local tags: ['Avahan/qwen2.5_0.5b-instruct-fc-Chat:latest', 'qwen2.5_0.5b-instruct-fc-Chat:dpo', 'qwen2.5_0.5b-instruct-fc-Chat:latest']
⚠️  no :sft tag: the SFT model was already overwritten earlier. Recreate it from outputs/sft_qwen2.5-0.5b/gguf if you still want to publish it.


## 6. Test it: tool call, no tool fits, plain chat
- **Tool request** with the full trained prompt (`build_messages`) → JSON call.
- **No tool fits** ("Tell me a joke" with only a weather tool) → `[]` (what DPO taught it; SFT called `get_weather`).
- **Plain chat** with **no** system prompt → normal text (works because no tool prompt is baked into the Modelfile).

In [6]:
record = {
    "query": "What's the weather like in Pune right now?",
    "tools": [{"name": "get_weather", "description": "Current weather for a city.",
               "parameters": {"city": {"type": "str", "description": "City name"}}}],
}
print("WITH TOOLS  :", ollama_chat(build_messages(record)))
print("NO TOOL FITS:", ollama_chat(build_messages({**record, "query": "Tell me a joke"})))
print("PLAIN CHAT  :", ollama_chat([{"role": "user", "content": "What is an LLM? One sentence."}])[:300])

WITH TOOLS  : [{"name": "get_weather", "arguments": {"city": "Pune"}}]
NO TOOL FITS: []
PLAIN CHAT  : An LLM is a large language model, which is a type of artificial intelligence that can generate human-like text.


## 7. Accuracy check: did GGUF/Q8 cost anything?
100 questions from the frozen test set through Ollama, scored with **the same scorer**, compared with the bf16 DPO model on the **same 100 questions** (`scored.csv` from notebook 10). A drop of ~1 point or less is normal for Q8_0.

In [7]:
import pandas as pd
from src.data_utils import read_jsonl
from src.evaluate import score_prediction

N = 100
test = read_jsonl(NOTEBOOKS / "data" / "eval" / "test_v1.jsonl")[:N]
gguf_em, slow, failed = [], [], []
t0 = time.time()
for i, r in enumerate(test):
    t = time.time()
    try:
        reply = ollama_chat(build_messages(r))
    except Exception as e:                      # one bad request shouldn't kill the whole run
        failed.append((r["id"], str(e)))
        reply = ""
    gguf_em.append(score_prediction(reply, r)["exact_match"])
    if time.time() - t > 20:                    # normal answers take ~1-3 s
        slow.append((r["id"], round(time.time() - t), reply[-120:]))
    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{N} done, {time.time() - t0:.0f}s")

bf16 = pd.read_csv(RESULTS_DIR / "scored.csv").set_index("id").loc[[r["id"] for r in test], "exact_match"]
print(f"\nExact match on {N} questions:  bf16 {100 * bf16.mean():.1f}%  |  GGUF {QUANT} {100 * sum(gguf_em) / N:.1f}%")
print("Slow answers (>20s):", slow or "none")
print("Failed requests:", failed or "none")

20/100 done, 50s
40/100 done, 102s
60/100 done, 152s
80/100 done, 203s
100/100 done, 253s

Exact match on 100 questions:  bf16 76.0%  |  GGUF q8_0 76.0%
Slow answers (>20s): none
Failed requests: none


## 8. Publish to ollama.com
Steps 5–7 only registered the model **on your PC**. To let anyone run `ollama run Avahan/qwen2.5_0.5b-instruct-fc-Chat`:

1. **Add this PC's public key** (one time): ollama.com → Settings → **Ollama keys** → Add, and paste the key printed below. The private half never leaves your PC; pushes are signed with it (same idea as SSH keys on GitHub).
2. Run the next cell: it pushes **`:sft`** (if kept), **`:dpo`** and **`:latest`**. The weights upload once per model (~530 MB each for SFT and DPO); extra tags are almost free. Progress lines appear every few seconds; each push is killed after 60 min.
3. Update the description on ollama.com: `:latest`/`:dpo` = SFT + DPO (answers `[]` when no tool fits), `:sft` = SFT only, plus a link to the Hugging Face model card for the prompt format and results.

In [8]:
pub_key = Path.home() / ".ollama" / "id_ed25519.pub"
print("Paste this into ollama.com -> Settings -> Ollama keys (only once):\n")
print(pub_key.read_text().strip() if pub_key.exists() else f"Not found at {pub_key}: has Ollama been run once on this PC?")

Paste this into ollama.com -> Settings -> Ollama keys (only once):

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIIu5NByerx+vPqCd5O6iASdfGk2EQiaMjn6rJmtJeFHB


In [9]:
models = ollama_models()
tags = [t for t in ("sft", TAG, "latest") if f"{OLLAMA_NAME}:{t}" in models]    # sft first, latest last
print("pushing tags:", tags)
for t in tags:
    run(["ollama", "cp", f"{OLLAMA_NAME}:{t}", f"{PUBLIC_NAME}:{t}"], timeout=120)   # namespaced copy
    run(["ollama", "push", f"{PUBLIC_NAME}:{t}"], timeout=3600, quiet_every=10)
    print(f"✅ pushed {PUBLIC_NAME}:{t}\n")
print(f"Published: https://ollama.com/{PUBLIC_NAME}/tags")
print(f"   ollama run {PUBLIC_NAME}        (= :{TAG}, SFT + DPO)")
if "sft" in tags:
    print(f"   ollama run {PUBLIC_NAME}:sft    (earlier SFT-only model)")

pushing tags: ['dpo', 'latest']
$ ollama cp qwen2.5_0.5b-instruct-fc-Chat:dpo Avahan/qwen2.5_0.5b-instruct-fc-Chat:dpo
   copied 'qwen2.5_0.5b-instruct-fc-Chat:dpo' to 'Avahan/qwen2.5_0.5b-instruct-fc-Chat:dpo'
   success
$ ollama push Avahan/qwen2.5_0.5b-instruct-fc-Chat:dpo
   retrieving manifest ⠋ retrieving manifest ⠙ retrieving manifest ⠹ [?
   pushing 85231dc6e84d   1% ▕                   ▏ 3.6 MB/531 MB  360 KB/s  24m23sretrieving manifest 
   pushing 85231dc6e84d   2% ▕                   ▏  11 MB/531 MB  821 KB/s  10m32sretrieving manifest 
   pushing 85231dc6e84d   4% ▕                   ▏  23 MB/531 MB  1.2 MB/s   6m53sretrieving manifest 
   pushing 85231dc6e84d   6% ▕█                  ▏  31 MB/531 MB  928 KB/s   8m57sretrieving manifest 
   pushing 85231dc6e84d   9% ▕█                  ▏  45 MB/531 MB  1.4 MB/s   5m41sretrieving manifest 
   pushing 85231dc6e84d  11% ▕██                 ▏  58 MB/531 MB  1.3 MB/s   5m57sretrieving manifest 
   pushing 85231dc6e84d  14% ▕██